# LLM per estrarre questioni giuridiche da sentenze tributarie

In [1]:
from openai import OpenAI
from openai import AsyncOpenAI
import os
import json
import sys
import pandas as pd
import asyncio
import time
from lxml import etree
import re
import logging
# Add the parent and grandparent directory to Python path
parent_dir = os.path.dirname(os.getcwd())  # Go up one level from regex_extract
sys.path.append(parent_dir)
grandparent_dir = os.path.dirname(parent_dir)  # /path/to
sys.path.append(grandparent_dir)
from utils.sent_paragraphs_position import sent_paragraphs_positions_bdgt 




In [ ]:
#select sentences to use for test

sentences_folder="/Users/k2482382/Documents/trib_data_unzipped/bdgt/scraping_bdgt_31_10_2025_txt_norm/"
#sentences_folder="./sentenze_validazione/"

#sentences_folder="/Users/k2482382/Documents/trib_data_unzipped/appello_incidentale/"

In [ ]:
df=pd.read_csv("../filter_sentenze/results/scraping_bdgt_31_10_2025.csv")
df=df[df['Anno']==2025]


In [ ]:
df

In [ ]:
df['Filename']=""
for idx, row in df.iterrows():
    if row['Text available']:
        filename=f"Sentenza_{row['Codice autorità emittente']}_{row['Numero provvedimento']}_{row['Anno']}.txt"
        df.at[idx,'Filename']=filename


In [ ]:
"""
df_107k=pd.read_csv("../filter_sentenze/results/scraping_bdgt_107k_2024.csv")
df_107k['Filename']=""
for idx, row in df_107k.iterrows():
    if row['Text available']:
        filename=f"Sentenza_{row['Codice autorità emittente']}_{row['Numero sentenza']}_{row['Anno']}.txt"
        df_107k.at[idx,'Filename']=filename
"""

In [ ]:
"""df = df[~df['Filename'].isin(df_107k['Filename'])]"""


In [2]:
prompt_questioni="""All'interno delle parentesi graffe è riportata una sentenza tributaria italiana. Il tuo compito è identificare ed estrarre le principali questioni giuridiche (generalmente una o due) che il giudice affronta e risolve per decidere la controversia. Considera esclusivamente le questioni su cui il giudice si pronuncia in modo esplicito e motivato.
Per ciascuna questione, svolgi le seguenti operazioni:
- Formula la questione giuridica in modo chiaro, autosufficiente e autonomo. La formulazione deve essere nella forma “Se + congiuntivo” e deve essere comprensibile da sola, senza dover consultare l’intera sentenza o altre questioni. Sii sintetico.
- Indica l’esito della questione, cioè la risposta al quesito giuridico. Includi solo la soluzione della questione senza indicare altre disposizioni del giudice.
- Scrivi un riassunto astrattivo di circa 100 parole della questione, che includa:
-- i fatti rilevanti per la questione
-- i riferimenti normativi e giurisprudenziali principali rilevanti per la questione
-- la sintesi del ragionamento del giudice per decidere la questione
-- la conclusione della questione raggiunta dal giudice
Il riassunto deve essere autonomo e comprensibile senza riferimento alla sentenza completa.

Usa sempre un linguaggio giuridico preciso. Includi solo le informazioni rilevanti per ciascuna questione.

Formato di output XML:
<questioni>
  <questione title="titolo questione">
    <text>testo della questione giuridica (~30 parole)</text>
    <esito_questione favorevole_a="ufficio|contribuente|other">testo esito</esito_questione>
    <riassunto> riassunto (~100 parole) della questione</riassunto>
  </questione>
</questioni>
 Genera solo il blocco XML e nulla oltre.

"""

prompt_ragionamento="""All'interno delle parentesi graffe è riportata una sentenza tributaria italiana. Tra parentesi quadre è inoltre fornita una questione giuridica già identificata all’interno della stessa sentenza. Il tuo compito è estrarre il ragionamento del giudice dalla sentenza in riferimento alla questione. Suddividi il ragionamento come segue:
- <premesse_fatto>: elenca le circostanze, azioni, eventi o dati del caso ritenuti rilevanti per la questione. Il fatto deve essere descritto con sufficiente precisione da permettere a un altro giudice di decidere una questione analoga in modo coerente.
- <lista_riferimenti_diritto>: elenca in modo completo TUTTI E SOLI i riferimenti giuridici (norme, giurisprudenza, principi generali, prassi amministrativa) usati dal giudice per risolvere la questione data, così come appaiono nella sentenza. Ordinali per importanza decrescente.
- <motivo_citazione>: per i soli riferimenti giuridici trattati più in dettaglio spiega perché il giudice li cita. Considera solo i riferimenti discussi in modo argomentato e approfondito nella sentenza.
- <ragionamento_giudice>: descrivi in testo libero come il giudice ha applicato i riferimenti giuridici al caso concreto, fino a arrivare alla decisione. Non includere le informazioni già presenti nelle premesse di fatto. Usa meno di 120 parole circa.
Usa sempre un linguaggio giuridico preciso.
Formato di output XML:
<premesse_fatto>
  <item>premessa di fatto 1</item>
  <item>premessa di fatto 2</item>
</premesse_fatto>
<lista_riferimenti_diritto>
  <item id="D1" type="jur" ref="Cassazione n. nnnn/yyyy"/>
  <item id="D2" type="norm" ref="art. xx comma z dlgs n. nnn/yyyy"/>
  <item id="D3" type="jur" ref="Cassazione n. nnnn/yyyy, Cassazione n. mmmm/yyyy"/>
</lista_riferimenti_diritto>
</motivo_citazione>
  <item id_ref="D1"> motivo citazione diritto </item>
  <item id_ref="D2"> motivo citazione diritto </item>
</motivo_citazione>
<ragionamento_giudice>testo ragionamento (meno di 150 parole)</ragionamento_giudice>
Nel campo ref dei riferimenti giuridici indica sempre gli estremi esatti dell’atto. Usa il campo type per classificare il riferimento:
- type="jur" per giurisprudenza. Se più sentenze vengono citate consecutivamente, raggruppale in un unico item;
- type="norm" per norme di legge;
- type="princ" per principi generali del diritto (in questo caso, ref contiene esclusivamente il nome del principio);
- type="prassi_amm" per la prassi amministrativa.
Includi solo le informazioni rilevanti per la questione data.
Genera solo il blocco XML e nulla oltre.

"""

prompt_monolitico="""All'interno delle parentesi graffe è riportata una sentenza tributaria italiana.
Il tuo compito è identificare ed estrarre le principali questioni giuridiche (una o massimo due) che il giudice affronta e risolve per decidere la controversia. Considera esclusivamente le questioni su cui il giudice si pronuncia in modo esplicito e motivato.
Per ciascuna questione, svolgi le seguenti operazioni:
- Formula la questione giuridica in modo chiaro, autosufficiente e autonomo. La formulazione deve essere nella forma “Se + congiuntivo” e deve essere comprensibile da sola, senza dover consultare l’intera sentenza o altre questioni.
- Indica l’esito della questione, cioè la risposta al quesito giuridico. 
- Scomponi il ragionamento del giudice nelle seguenti parti: 
-- <premesse_fatto>: elenca le circostanze, azioni, eventi o dati del caso ritenuti rilevanti per la questione.
-- <lista_riferimenti_diritto>: elenca in modo completo TUTTI i riferimenti giuridici (norme, giurisprudenza, principi generali, prassi amministrativa) rilevanti per la questione, così come appaiono nella sentenza. Ordinali per importanza decrescente.
-- <motivo_citazione>: per i soli riferimenti giuridici trattati più in dettaglio spiega perché il giudice li cita. Considera solo i riferimenti discussi in modo argomentato e approfondito nella sentenza.
-- <ragionamento_giudice>: descrivi in testo libero come il giudice ha applicato i riferimenti giuridici al caso concreto, fino a arrivare alla decisione. Non includere le informazioni già presenti nelle premesse di fatto.
- Redigi un riassunto astrattivo della questione, tra 100 e 150 parole, che includa: fatto, diritto, ragionamento e conclusione. Il riassunto deve essere autosufficiente e completo, comprensibile senza fare riferimento alla sentenza completa o ad altre questioni.
Requisiti generali:
- Ogni questione deve essere autonoma, autosufficiente e formulata in linguaggio giuridico preciso e dettagliato.
- Il fatto deve essere descritto con sufficiente precisione da permettere a un altro giudice di decidere una questione analoga in modo coerente.


Formato di output richiesto: esclusivamente in XML, come nello schema seguente:
<questioni>
  <questione title="titolo questione">
    <text>testo della questione giuridica</text>
    <esito_questione favorevole_a="ufficio|contribuente|other">testo esito</esito_questione>
    <premesse_fatto>
      <item>premessa di fatto 1</item>
      <item>premessa di fatto 2</item>
    </premesse_fatto>
    <lista_riferimenti_diritto>
      <item id="D1" type="jur" ref="Cassazione n. nnnn/yyyy"/>
      <item id="D2" type="norm" ref="art. xx comma z d.lgs n. nnn/yyyy"/>
    </lista_riferimenti_diritto>
    </motivo_citazione>
      <item id_ref="D1"> motivo citazione diritto </item>
      <item id_ref="D2"> motivo citazione diritto </item>
    </motivo_citazione>
    <ragionamento_giudice>testo ragionamento (~100 parole)</ragionamento_giudice>
    <riassunto>testo del riassunto (~100 parole)</riassunto>
  </questione>
</questioni>
Nel campo ref dei riferimenti giuridici indica sempre gli estremi esatti dell’atto. Usa il campo type per classificare il riferimento:
- type="jur" per giurisprudenza. Se più sentenze vengono citate consecutivamente, raggruppale in un unico item;
- type="norm" per norme di legge;
- type="princ" per principi generali del diritto (in questo caso, ref contiene esclusivamente il nome del principio);
- type="prassi_amm" per la prassi amministrativa.
Genera solo il blocco XML e nulla oltre.

"""

prompt_monolitico="""All'interno delle parentesi graffe è riportata una sentenza tributaria italiana.
Il tuo compito è identificare ed estrarre le principali questioni giuridiche (una o massimo due) che il giudice affronta e risolve per decidere la controversia. Considera esclusivamente le questioni su cui il giudice si pronuncia in modo esplicito e motivato.
Per ciascuna questione, svolgi le seguenti operazioni:
- Formula la questione giuridica in modo chiaro, autosufficiente e autonomo. La formulazione deve essere nella forma “Se + congiuntivo” e deve essere comprensibile da sola, senza dover consultare l’intera sentenza o altre questioni.
- Indica l’esito della questione, cioè la risposta al quesito giuridico. 
- Scomponi il ragionamento del giudice nelle seguenti parti: 
-- <premesse_fatto>: elenca le circostanze, azioni, eventi o dati del caso ritenuti rilevanti per la questione.
-- <lista_riferimenti_diritto>: elenca in modo completo TUTTI i riferimenti giuridici (norme, giurisprudenza, principi generali, prassi amministrativa) rilevanti per la questione, così come appaiono nella sentenza. Ordinali per importanza decrescente.
-- <motivo_citazione>: per i soli riferimenti giuridici trattati più in dettaglio spiega perché il giudice li cita. Considera solo i riferimenti discussi in modo argomentato e approfondito nella sentenza.
-- <ragionamento_giudice>: descrivi in testo libero come il giudice ha applicato i riferimenti giuridici al caso concreto, fino a arrivare alla decisione. Non includere le informazioni già presenti nelle premesse di fatto.
- Redigi un riassunto astrattivo della questione, tra 100 e 150 parole, che includa: fatto, diritto, ragionamento e conclusione. Il riassunto deve essere autosufficiente e completo, comprensibile senza fare riferimento alla sentenza completa o ad altre questioni.
Requisiti generali:
- Ogni questione deve essere autonoma, autosufficiente e formulata in linguaggio giuridico preciso e dettagliato.
- Il fatto deve essere descritto con sufficiente precisione da permettere a un altro giudice di decidere una questione analoga in modo coerente.


Formato di output richiesto: esclusivamente in XML, come nello schema seguente:
<questioni>
  <questione title="titolo questione">
    <text>testo della questione giuridica</text>
    <esito_questione favorevole_a="ufficio|contribuente|other">testo esito</esito_questione>
    <premesse_fatto>
      <item>premessa di fatto 1</item>
      <item>premessa di fatto 2</item>
    </premesse_fatto>
    <lista_riferimenti_diritto>
      <item id="D1" type="jur" ref="Cassazione n. nnnn/yyyy"/>
      <item id="D2" type="norm" ref="art. xx comma z d.lgs n. nnn/yyyy"/>
    </lista_riferimenti_diritto>
    </motivo_citazione>
      <item id_ref="D1"> motivo citazione diritto </item>
      <item id_ref="D2"> motivo citazione diritto </item>
    </motivo_citazione>
    <ragionamento_giudice>testo ragionamento (~100 parole)</ragionamento_giudice>
    <riassunto>testo del riassunto (~100 parole)</riassunto>
  </questione>
</questioni>
Nel campo ref dei riferimenti giuridici indica sempre gli estremi esatti dell’atto. Usa il campo type per classificare il riferimento:
- type="jur" per giurisprudenza. Se più sentenze vengono citate consecutivamente, raggruppale in un unico item;
- type="norm" per norme di legge;
- type="princ" per principi generali del diritto (in questo caso, ref contiene esclusivamente il nome del principio);
- type="prassi_amm" per la prassi amministrativa.
Genera solo il blocco XML e nulla oltre."""

prompt_questioni="""All'interno delle parentesi graffe è riportata una sentenza tributaria italiana. Il tuo compito è identificare ed estrarre le principali questioni giuridiche (generalmente una o due) che il giudice affronta e risolve per decidere la controversia. Considera esclusivamente le questioni su cui il giudice si pronuncia in modo esplicito e motivato.
Per ciascuna questione, svolgi le seguenti operazioni:
- Formula la questione giuridica in modo chiaro, autosufficiente e autonomo. La formulazione deve essere nella forma “Se + congiuntivo” e deve essere comprensibile da sola, senza dover consultare l’intera sentenza o altre questioni. Sii sintetico.
- Indica l’esito della questione, cioè la risposta al quesito giuridico. Includi solo la soluzione della questione senza indicare altre disposizioni del giudice.
- Scrivi un riassunto astrattivo di circa 100 parole della questione, che includa:
-- i fatti rilevanti per la questione
-- i riferimenti normativi e giurisprudenziali principali rilevanti per la questione
-- la sintesi del ragionamento del giudice per decidere la questione
-- la conclusione della questione raggiunta dal giudice
Il riassunto deve essere autonomo e comprensibile senza riferimento alla sentenza completa.

Usa sempre un linguaggio giuridico preciso. Includi solo le informazioni rilevanti per ciascuna questione.

Formato di output XML:
<questioni>
  <questione title="titolo questione">
    <text>testo della questione giuridica (~30 parole)</text>
    <esito_questione favorevole_a="ufficio|contribuente|other">testo esito</esito_questione>
    <riassunto> riassunto (~100 parole) della questione</riassunto>
  </questione>
</questioni>
 Genera solo il blocco XML e nulla oltre."""

prompt_ragionamento="""All'interno delle parentesi graffe è riportata una sentenza tributaria italiana. Tra parentesi quadre è inoltre fornita una questione giuridica già identificata all’interno della stessa sentenza. Il tuo compito è estrarre il ragionamento del giudice dalla sentenza in riferimento alla questione. Suddividi il ragionamento come segue:
- <premesse_fatto>: elenca le circostanze, azioni, eventi o dati del caso ritenuti rilevanti per la questione. Il fatto deve essere descritto con sufficiente precisione da permettere a un altro giudice di decidere una questione analoga in modo coerente.
- <lista_riferimenti_diritto>: elenca in modo completo TUTTI E SOLI i riferimenti giuridici (norme, giurisprudenza, principi generali, prassi amministrativa) usati dal giudice per risolvere la questione data, così come appaiono nella sentenza. Ordinali per importanza decrescente.
- <motivo_citazione>: per i soli riferimenti giuridici trattati più in dettaglio spiega perché il giudice li cita. Considera solo i riferimenti discussi in modo argomentato e approfondito nella sentenza.
- <ragionamento_giudice>: descrivi in testo libero come il giudice ha applicato i riferimenti giuridici al caso concreto, fino a arrivare alla decisione. Non includere le informazioni già presenti nelle premesse di fatto. Usa meno di 120 parole circa.
Usa sempre un linguaggio giuridico preciso.
Formato di output XML:
<premesse_fatto>
  <item>premessa di fatto 1</item>
  <item>premessa di fatto 2</item>
</premesse_fatto>
<lista_riferimenti_diritto>
  <item id="D1" type="jur" ref="Cassazione n. nnnn/yyyy"/>
  <item id="D2" type="norm" ref="art. xx comma z dlgs n. nnn/yyyy"/>
  <item id="D3" type="jur" ref="Cassazione n. nnnn/yyyy, Cassazione n. mmmm/yyyy"/>
</lista_riferimenti_diritto>
</motivo_citazione>
  <item id_ref="D1"> motivo citazione diritto </item>
  <item id_ref="D2"> motivo citazione diritto </item>
</motivo_citazione>
<ragionamento_giudice>testo ragionamento (meno di 150 parole)</ragionamento_giudice>
Nel campo ref dei riferimenti giuridici indica sempre gli estremi esatti dell’atto. Usa il campo type per classificare il riferimento:
- type="jur" per giurisprudenza. Se più sentenze vengono citate consecutivamente, raggruppale in un unico item;
- type="norm" per norme di legge;
- type="princ" per principi generali del diritto (in questo caso, ref contiene esclusivamente il nome del principio);
- type="prassi_amm" per la prassi amministrativa.
Includi solo le informazioni rilevanti per la questione data.
Genera solo il blocco XML e nulla oltre.
"""

In [3]:
def clean_xml(xml,start_tag,end_tag):
    for i in range (len(xml)):
        if xml[i:i+len(start_tag)]==start_tag:
            xml=xml[i:]
            break
    for i in range (len(xml)):
        pos=len(xml)-i-1
        if xml[pos-len(end_tag):pos]==end_tag:
            xml=xml[:pos]
            break
   
    return xml

In [4]:
def prompt_to_use(sent_text):
    """
    Choose the prompt to use based on the length of the sentence.
    retunrs zero for no prompt (sentenza too short)
    return one for monolithic prompt, two for multistep prompt.
    """
    
    dict_paragraphs_positions=sent_paragraphs_positions_bdgt(sent_text)
    len_motivi=None
    len_svolgimento_to_end=None
    len_contenuto=None
    if dict_paragraphs_positions['motivi della decisione'] is not None and dict_paragraphs_positions['p.q.m.'] is not None:
        start_pos=dict_paragraphs_positions['motivi della decisione'][0]
        end_pos=dict_paragraphs_positions['p.q.m.'][0]
        len_motivi=end_pos-start_pos

    if dict_paragraphs_positions['svolgimento del processo'] is not None:
        start_pos_svolgimento=dict_paragraphs_positions['svolgimento del processo'][0]
        len_svolgimento_to_end=len(sent_text)-start_pos_svolgimento

    if dict_paragraphs_positions['richieste delle parti'] is not None:
        end_pos_richieste=dict_paragraphs_positions['richieste delle parti'][1]
        len_contenuto=len(sent_text)-end_pos_richieste

    if dict_paragraphs_positions['pubblica udienza'] is not None:
        end_pos_pubblica_udienza=dict_paragraphs_positions['pubblica udienza'][1]
        if len_contenuto is None:
            len_contenuto=len(sent_text)-end_pos_pubblica_udienza
        else:
            len_contenuto=min(len_contenuto,len(sent_text)-end_pos_pubblica_udienza)

    if dict_paragraphs_positions['camera di consiglio'] is not None:
        end_pos_camera_di_consiglio=dict_paragraphs_positions['camera di consiglio'][1]
        if len_contenuto is None:
            len_contenuto=len(sent_text)-end_pos_camera_di_consiglio
        else:
            len_contenuto=min(len_contenuto,len(sent_text)-end_pos_camera_di_consiglio)

    result=2
    #print(len_motivi,len_svolgimento_to_end,len_contenuto)
    #if the motivazione is more than 3600 characters, use prompt 2. If it is less than 680 characters, discard the sentence
    if len_motivi is not None:
        if len_motivi<4100:
            result=1
        if len_motivi<680:
            result=0
        #if the svolgimento to end is more than 12000 characters, use prompt 2 (overriding the length of the motivazione)
        if len_svolgimento_to_end is not None:
            if len_svolgimento_to_end>12000:
                result=2
    #if the the svolgimento to end is less than 5800 characters, use prompt 1. If it is less than 1400 characters, discard the sentence
    elif len_svolgimento_to_end is not None:
        if len_svolgimento_to_end<7000:
            result=1
        if len_svolgimento_to_end<1400:
            result=0
    #if the content (dopo intestazione, parti e atti impugnati) of the sentence is less than 5800 characters, use prompt 1. If it is less than 1400 characters, discard the sentence
    elif len_contenuto is not None:
        if len_contenuto<7000:
            result=1
        if len_contenuto<1400:
            result=0
    #if the whole sentence is less than 7000 characters, use prompt 1. If it is less than 2300 characters, discard the sentence
    else:
        if len(sent_text)<7600:
            result=1
        elif len(sent_text)<2300:
            result=0
    return result


In [5]:
def add_ids_to_questioni(tree) -> etree._ElementTree:
    "adds sequential ids to each questione"
    root = tree.getroot()
    
    # Add id attributes to <questione> elements
    for i, questione in enumerate(root.findall('questione'), start=1):
        questione.set('id', f"Q{i}")

    return tree

#sometimes there are characters like '&,<,>' which must be escaped in xml. One does so with this function 
def escape_xml_content(text):
    """
    Escape all problematic characters in XML content to prevent parsing errors.
    
    Args:
        text (str): Raw text content that may contain XML-unsafe characters
        
    Returns:
        str: XML-safe escaped text
    """
    if not text:
        return text
    
    # Step 1: Handle ampersands first (must be done before other replacements)
    # Only escape & that aren't already part of valid XML entities
    text = re.sub(r'&(?!(?:amp|lt|gt|quot|apos|#\d+|#x[0-9a-fA-F]+);)', '&amp;', text)
    
    # Step 2: Escape other XML special characters
    replacements = {
        '<': '&lt;',
        '>': '&gt;',
        '"': '&quot;',
        "'": '&apos;'
    }
    
    for char, escape_seq in replacements.items():
        text = text.replace(char, escape_seq)
    
    # Step 3: Remove invalid XML control characters
    # XML 1.0 only allows: tab (0x09), newline (0x0A), carriage return (0x0D), 
    # and characters >= 0x20 (except 0x7F-0x9F range)
    text = re.sub(r'[\x00-\x08\x0B\x0C\x0E-\x1F\x7F-\x9F]', '', text)
    
    return text

def clean_xml_tree(tree_or_element):
    """
    Apply escape_xml_content recursively to all text and attributes in an XML tree.
    Modifies the tree in-place and returns it.
    """
    # Get the root element
    if hasattr(tree_or_element, 'getroot'):
        root = tree_or_element.getroot()
    else:
        root = tree_or_element
    
    def clean_element(element):
        # Clean text content
        if element.text:
            element.text = escape_xml_content(element.text)
            
        # Clean tail text
        if element.tail:
            element.tail = escape_xml_content(element.tail)
            
        # Clean attributes
        for attr_name, attr_value in element.attrib.items():
            element.attrib[attr_name] = escape_xml_content(attr_value)
        
        # Recursively clean children
        for child in element:
            clean_element(child)
    
    clean_element(root)
    return tree_or_element


## Example with single sentence

In [6]:
def ask_model(client,prompt, sentenza, model,questione="",seed=42, temp=0.0,verbose=0,specify_quantization=False ):
    messages = [
        {"role": "system", "content": ""}, #Sei un esperto di diritto tributario italiano.
        {"role": "user", "content": prompt+questione+ "\n\n{" + sentenza.strip() + "}"},
    ]
    extra_body={}
    if specify_quantization:
        extra_body={
        "provider": {
            "quantizations": ["fp8"]
        }
    }
    response = client.chat.completions.create(
        model=model,
        messages=messages,
        max_tokens=4096,
        temperature=temp,
        stream=False,
        seed=seed,
        extra_body=extra_body
    )
    return response.choices[0].message.content.strip()

In [22]:
#deepseek
#client = OpenAI(api_key=os.environ["OPENROUTER_API_KEY"], base_url="https://api.deepseek.com")
#openrouter
client = OpenAI(api_key=os.environ["OPENROUTER_API_KEY"], base_url="https://openrouter.ai/api/v1")


In [39]:
sentences_folder="/Users/k2482382/Documents/trib_data_unzipped/bdgt/scraping_bdgt_2021_txt_norm/"

In [42]:
#Sentenza_U91_11131_2023.json #Sentenza_U91_9814_2021.txt#Sentenza_V12_962_2022.txt
with open(sentences_folder+"Sentenza_U91_9814_2021.txt",'r') as sent_file: #Sentenza_V01_136_2024 #Sentenza_V36_1985_2021.txt
    sentenza=sent_file.read()
model='deepseek/deepseek-chat-v3-0324'#'google/gemini-3-pro-preview'#'deepseek/deepseek-chat-v3-0324" #"moonshotai/kimi-k2"# "meta-llama/llama-3.3-70b-instruct"#
print(prompt_questioni)

int_prompt=prompt_to_use(sentenza)
if int_prompt==0:
    print("Sentence too short, no prompt to use")
elif int_prompt==1:
    print("Using monolithic prompt")
    questioni = ask_model(client,prompt_monolitico, sentenza, model=model)
elif int_prompt==2:
    print("Using multistep prompt")
    questioni = ask_model(client,prompt_questioni, sentenza, model=model)

print(questioni)
questioni=clean_xml(questioni,"<questioni>","</questioni>")

root_questioni = etree.fromstring(questioni.encode())
  

All'interno delle parentesi graffe è riportata una sentenza tributaria italiana. Il tuo compito è identificare ed estrarre le principali questioni giuridiche (generalmente una o due) che il giudice affronta e risolve per decidere la controversia. Considera esclusivamente le questioni su cui il giudice si pronuncia in modo esplicito e motivato.
Per ciascuna questione, svolgi le seguenti operazioni:
- Formula la questione giuridica in modo chiaro, autosufficiente e autonomo. La formulazione deve essere nella forma “Se + congiuntivo” e deve essere comprensibile da sola, senza dover consultare l’intera sentenza o altre questioni. Sii sintetico.
- Indica l’esito della questione, cioè la risposta al quesito giuridico. Includi solo la soluzione della questione senza indicare altre disposizioni del giudice.
- Scrivi un riassunto astrattivo di circa 100 parole della questione, che includa:
-- i fatti rilevanti per la questione
-- i riferimenti normativi e giurisprudenziali principali rilevanti 

In [43]:
# --- Passo 2 e 3 per ciascuna questione ---
# Output XML finale
if int_prompt==2:
    final_xml = etree.Element("questioni")
    for questione in root_questioni.xpath("//questione"):
        print("questione",etree.tostring(root_questioni, pretty_print=True).decode())
        titolo = questione.get("title")
        testo = questione.findtext("text")
        riassunto = questione.find("riassunto")

        esito = questione.find("esito_questione")

        nuova_q = etree.Element("questione", title=titolo)
        etree.SubElement(nuova_q, "text").text = testo
        nuova_q.append(esito)
        nuova_q.append(riassunto)

        # Passo 2: ragionamento
        testo_input_questione = f"QUESTIONE\n[{titolo}\n{testo}\n ESITO: {esito.text}]"
        output_ragionamento = ask_model(client, prompt_ragionamento, sentenza,model, testo_input_questione)
        #print(output_riassunto)
        output_ragionamento="\n"+clean_xml(output_ragionamento,"<premesse_fatto>","</ragionamento_giudice>")
        print(output_ragionamento)
        ragionamento_elem = etree.fromstring(f"<wrapper>{output_ragionamento}</wrapper>".encode())
        for tag in ["premesse_fatto","lista_riferimenti_diritto","motivo_citazione","ragionamento_giudice"]:
            elem = ragionamento_elem.find(tag)
            if elem is not None:
                nuova_q.append(elem)
        final_xml.append(nuova_q)
if int_prompt==1:
    tree=etree.ElementTree(root_questioni)
elif int_prompt==2:
    tree = etree.ElementTree(final_xml)
    
tree=add_ids_to_questioni(tree)
etree.indent(tree, space="  ")  # forza indentazione leggibile
tree.write("./results/output_test_b1.xml", pretty_print=True, xml_declaration=True, encoding="utf-8")
print("✅ output file generato con successo.")

✅ output file generato con successo.


## Check XML correctness


In [ ]:
import xml.etree.ElementTree as ET

with open('./results/output_test2.xml', 'r') as outfile_xml:
    sent_xml=outfile_xml.read()
try:
    ET.fromstring(sent_xml)
    print("XML ben formato!")
except ET.ParseError as e:
    print(f"Errore di parsing XML: {e}")

XML ben formato!


In [18]:
from lxml import etree
# Carica il file XSD
xsd_tree = etree.XMLSchema(etree.parse("./schemas/reasoning_schema_multistep_v5_9.xsd"))

# Carica il file XML da controllare
xml_tree = etree.parse("./results/output_test.xml")

# Valida
if xsd_tree.validate(xml_tree):
    print("XML valido secondo lo schema XSD!")
else:
    print("Errore nella validazione XML:", xsd_tree.error_log)

XML valido secondo lo schema XSD!


# Concurrent execution

In [ ]:
#start logger
logging.basicConfig(
    filename="logfile_extr_questioni.log",
    encoding="utf-8",
    filemode="a",
    format="{asctime} - {levelname} - {message}",
    style="{",
    datefmt="%Y-%m-%d %H:%M",
    level=logging.INFO,
    force=True
 )
logging.info("Starting XML reasoning validation with multistep prompt v5")
for handler in logging.root.handlers:
    handler.flush()


In [ ]:
# Function to read a file
def read_file(filename, folder):
    with open(folder+filename, 'r') as file:
        return file.read()

async def ask_model(client,prompt, sentenza, model,questione="",seed=42, temp=0.0,verbose=0):

    messages = [
        {"role": "system", "content": ""}, #Sei un esperto di diritto tributario italiano.
        {"role": "user", "content": prompt+questione+ "\n\n{" + sentenza.strip() + "}"},
    ]
    response = await client.chat.completions.create(
        model=model,
        messages=messages,
        max_tokens=4096,
        temperature=temp,
        stream=False,
        seed=seed,
    extra_body={
    "provider": {
        "quantizations": ["fp8"]
    }
    }
    )
    return response.choices[0].message.content.strip()


async def worker(client, queue, sentence_folder, worker_idx, model, outfolder_xml,xsd_tree):
    while not queue.empty():
        # Get the next filename from the queue
        await asyncio.sleep(0.5)
        start_time = time.time()
        filename = await queue.get()
        
        try:
            # Add timeout for entire file processing (10 minutes)
            async with asyncio.timeout(600):
                # Read the file content
                sentenza = read_file(filename, sentence_folder)
                int_prompt = prompt_to_use(sentenza)
                if int_prompt == 0:
                    print(f"W{worker_idx}  SKIPPED {filename} - sentence too short")
                    logging.info(f"file {filename} SKIPPED  - sentence too short")
                    continue
                elif int_prompt == 1:
                    logging.info(f"file {filename} using monolithic prompt")
                    print(f"W{worker_idx} using monolithic prompt for {filename}")
                    questioni = await ask_model(client, prompt_monolitico, sentenza, model=model)
                elif int_prompt == 2:
                    logging.info(f"file {filename} using multistep prompt")
                    print(f"W{worker_idx} using multistep prompt for {filename}")
                    questioni = await ask_model(client, prompt_questioni, sentenza, model=model)
                questioni= re.sub(r'&(?!(?:amp|lt|gt|quot|apos|#\d+|#x[0-9a-fA-F]+);)', '&amp;', questioni)
                questioni = clean_xml(questioni, "<questioni>", "</questioni>")
                root_questioni = etree.fromstring(questioni.encode())
                print(f"W{worker_idx} file: {filename}, time {time.time()-start_time:.1f} s. Got questione XML response", flush=True)
                if int_prompt == 2:
                    final_xml = etree.Element("questioni")
                    for idx, questione in enumerate(root_questioni.xpath("//questione")):
                        titolo = questione.get("title")
                        testo = questione.findtext("text")
                        riassunto = questione.find("riassunto")
                        esito = questione.find("esito_questione")

                        nuova_q = etree.Element("questione", title=titolo)
                        etree.SubElement(nuova_q, "text").text = testo
                        nuova_q.append(esito)

                        # Passo 2: ragionamento with retry limit
                        testo_input_questione = f"QUESTIONE\n[{titolo}\n{testo}\n ESITO: {esito.text}]"
                        flag_ragionamento = False
                        retry_count = 0
                        max_retries = 2
                        required_tags = ["<premesse_fatto>", "<lista_riferimenti_diritto>", "<motivo_citazione>","<ragionamento_giudice>"]
                        
                        while not flag_ragionamento and retry_count < max_retries:
                            try:
                                print(f"W{worker_idx}: Q{idx} sending prompt ragionamento for {filename} (attempt {retry_count + 1}/{max_retries})...")
                                start_time = time.time()
                                output_ragionamento = await ask_model(client, prompt_ragionamento, sentenza, model, testo_input_questione)
                                output_ragionamento = re.sub(r'&(?!(?:amp|lt|gt|quot|apos|#\d+|#x[0-9a-fA-F]+);)', '&amp;', output_ragionamento)

                                print(f"W{worker_idx} file: {filename}, time {time.time()-start_time:.1f} s. Got ragionamento XML response", flush=True)
                                output_ragionamento = "\n" + clean_xml(output_ragionamento, "<premesse_fatto>", "</ragionamento_giudice>")
                                
                                if all(tag in output_ragionamento for tag in required_tags):
                                    flag_ragionamento = True
                                else:
                                    missing_tags = [tag for tag in required_tags if tag not in output_ragionamento]
                                    print(f"W{worker_idx} Q{idx} missing tags: {missing_tags}")
                                    
                            except Exception as e:
                                print(f"W{worker_idx} Q{idx} ragionamento attempt {retry_count + 1} failed: {e}")
                            
                            retry_count += 1
                            
                            if not flag_ragionamento and retry_count < max_retries:
                                print(f"W{worker_idx} Q{idx} retrying ragionamento in 1 second...")
                                await asyncio.sleep(1)
                        
                        # Only process if we got valid ragionamento
                        if flag_ragionamento:
                            ragionamento_elem = etree.fromstring(f"<wrapper>{output_ragionamento}</wrapper>".encode())
                            for tag in ["premesse_fatto", "lista_riferimenti_diritto", "motivo_citazione", "ragionamento_giudice"]:
                                elem = ragionamento_elem.find(tag)
                                if elem is not None:
                                    nuova_q.append(elem)
                        else:
                            print(f"W{worker_idx} Q{idx} failed to get valid ragionamento after {max_retries} attempts")
                        nuova_q.append(riassunto)
                        final_xml.append(nuova_q)

                    # Write xml to file (moved outside the loop)
                    tree = etree.ElementTree(final_xml)
                elif int_prompt == 1:
                    tree = etree.ElementTree(root_questioni)
                tree=add_ids_to_questioni(tree)

                if xsd_tree.validate(tree):
                    print("XML valido secondo lo schema XSD!")
                else:
                    raise Exception(f"Errore nella validazione XML: {xsd_tree.error_log}")
                
                etree.indent(tree, space="  ")
                filepath_xml = outfolder_xml + filename[:-4] + ".xml"
                tree.write(filepath_xml, pretty_print=True, xml_declaration=True, encoding="utf-8")
                #print(f"W{worker_idx} ✓ SUCCESSFULLY processed {filename}")

        except asyncio.TimeoutError:
            print(f"W{worker_idx} ✗ TIMEOUT processing {filename} after 10 minutes")
            logging.error(f"File {filename} processing timed out after 10 minutes")
        except asyncio.CancelledError:
            print(f"W{worker_idx} ✗ CANCELLED while processing {filename}")
            logging.error(f"File {filename} processing was cancelled")
            raise  # Re-raise to properly handle cancellation
        except Exception as e:
            print(f"W{worker_idx} ✗ ERROR processing {filename}: {e}")
            logging.error(f"File {filename} error processing: {e}")
        finally:
            # Always mark the task as done
            queue.task_done()
            print(f"W{worker_idx} finished with {filename}, queue size: {queue.qsize()}")

In [ ]:
print(sentences_folder)

In [ ]:
#client = AsyncOpenAI(api_key=os.environ["OPENROUTER_API_KEY"], base_url="https://api.deepseek.com") #meta-llama/llama-3.3-70b-instruct
client = AsyncOpenAI(api_key=os.environ["OPENROUTER_API_KEY"], base_url="https://openrouter.ai/api/v1")

model="deepseek/deepseek-chat-v3-0324"#'mistralai/devstral-2512'#"deepseek/deepseek-v3.2-exp"#"deepseek/deepseek-chat-v3.1"#"moonshotai/kimi-k2"#"deepseek-chat"#"meta-llama/llama-3.3-70b-instruct" #"deepseek-chat"#"qwen/qwq-32b" #"google/gemma-3-27b-it" #"deepseek-chat"
#outfolder_xml="./results/xml_reasoning_validazione_v5_14_dsv30324/"
outfolder_xml="/Users/k2482382/Documents/trib_data_unzipped/bdgt/xml_31_10_2025/"
os.makedirs(outfolder_xml, exist_ok=True)
NUM_WORKERS=120
filenames=df['Filename'].tolist()
#filenames=[fname for fname in os.listdir(sentences_folder) if fname.endswith(".txt")]

filenames_xml=[fname[:-4]+".txt" for fname in os.listdir(outfolder_xml)]
#filenames_json=[fname[:-5]+".txt" for fname in os.listdir(outfolder_json)]
xsd_tree = etree.XMLSchema(etree.parse("./schemas/reasoning_schema_multistep_v5_9.xsd"))

#for fname_json in filenames_json:
for fname_xml in filenames_xml:
    print(f" Removing {fname_xml} from list")
    if fname_xml in filenames:
        filenames.remove(fname_xml)
print(f" Filenames to be processed: {len(filenames)}" )

async def run_queue():
    # Create a queue and add all filenames to it
    queue = asyncio.Queue()
    for filename in filenames:
        queue.put_nowait(filename)
    
    print(f"Starting processing with {NUM_WORKERS} workers, {queue.qsize()} files in queue")
    
    # Create worker tasks
    workers = [asyncio.create_task(worker(client, queue, sentences_folder, idx, model, outfolder_xml,xsd_tree)) for idx in range(NUM_WORKERS)]
    
    try:
        # Wait until the queue is fully processed with a timeout
        await asyncio.wait_for(queue.join(), timeout=360000)  # 100 hour timeout
        print("All files processed successfully!")
    except asyncio.TimeoutError:
        print("Queue processing timed out after 100 hours")
    finally:
        # Cancel the worker tasks
        print("Cancelling workers...")
        for w in workers:
            w.cancel()
        
        # Wait for workers to finish cancellation
        await asyncio.gather(*workers, return_exceptions=True)
        print("All workers cancelled")


In [ ]:
# Run the asyncio event loop
await run_queue()

In [ ]:
outfolder_xml

In [ ]:
count_questioni=0
tot_len=0
for fname in [fname for fname in os.listdir(outfolder_xml) if fname.endswith('.xml')][:100]:
    xml_tree = etree.parse(outfolder_xml+fname)
    root=xml_tree.getroot()
    for questione in root:
        count_questioni+=1
        print(count_questioni,'||',questione.find('text').text)
        tot_len+=len(questione.find('text').text)

    



    
    


In [ ]:
tot_len/count_questioni

In [ ]:
count_questioni=0
tot_len=0
for fname in [fname for fname in os.listdir("/Users/k2482382/Documents/trib_data_unzipped/run_100k_sent_2024_p10v5_ds31/") if fname.endswith('.xml')][:100]:
    xml_tree = etree.parse("/Users/k2482382/Documents/trib_data_unzipped/run_100k_sent_2024_p10v5_ds31/"+fname)
    root=xml_tree.getroot()
    for questione in root:
        count_questioni+=1
        tot_len+=len(questione.find('text').text)
        print(count_questioni,'||',questione.find('text').text)



In [ ]:
tot_len/count_questioni

In [ ]:
len("Se l'avviso di accertamento, che si limitava a recuperare l'imposta non versata sulla base dei valori precedentemente dichiarati dal contribuente senza modificarli, fosse adeguatamente motivato.")